# Wikipedia Data Pipeline
This notebook contains the `WikipediaPipeline` class, which handles crawling Wikipedia articles from a specific category, preprocessing the text (tokenization, stop-word removal, and stemming), and saving both the raw text files and a generated inverted index.

In [12]:
%pip install Wikipedia-API


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
%pip install nltk


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [14]:
import wikipediaapi
import json
import os
import re
import math
import collections
from concurrent.futures import ThreadPoolExecutor
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import nltk

# Ensure required NLTK components are downloaded
nltk.download('stopwords', quiet=True)

True

## Pipeline Definition

In [ ]:
class WikipediaPipeline:
    def __init__(self, category="Category:Computer_science", limit=2050):
        # User Agent is required by Wikipedia API policy
        self.wiki = wikipediaapi.Wikipedia('CS575Project/2.0 (contact@example.com)', 'en')
        self.start_category = category
        self.limit = limit
        self.data_path = 'data/'
        self.stemmer = PorterStemmer()
        self.stop_words = set(stopwords.words('english'))
        
        if not os.path.exists(self.data_path):
            os.makedirs(self.data_path)
            
    def preprocess(self, text):
        # Convert to lowercase and split text into clean alphanumeric tokens
        tokens = re.findall(r'\w+', text.lower())
        # Filter out common stop words and reduce words to their base stem
        return [self.stemmer.stem(t) for t in tokens if t not in self.stop_words]
        
    def run(self):
        queue = collections.deque([self.start_category])
        visited_categories = {self.start_category}
        indexed_pages = {} # {page_id: page_title}
        index = {}         # Inverted Index: {term: {doc_id: count}}
        doc_lengths = {}   # Track text length for normalization metrics
        
        print("Executing recursive breadth-first traversal across categories...")
        
        while queue and len(indexed_pages) < self.limit:
            current_cat_name = queue.popleft()
            cat_page = self.wiki.page(current_cat_name)
            
            # Guard against invalid or broken category strings
            if not cat_page.exists():
                continue
                
            for member in cat_page.categorymembers.values():
                if len(indexed_pages) >= self.limit: 
                    break
                
                # If the item is a Subcategory, append it to the BFS queue
                if member.ns == wikipediaapi.Namespace.CATEGORY:
                    if member.title not in visited_categories:
                        visited_categories.add(member.title)
                        queue.append(member.title)
                
                # If the item is a standard Main Article page, index it
                elif member.ns == wikipediaapi.Namespace.MAIN:
                    if member.pageid not in indexed_pages:
                        page_data = self.wiki.page(member.title)
                        
                        if page_data.exists() and len(page_data.text.strip()) > 100:
                            tokens = self.preprocess(page_data.text)
                            if len(tokens) == 0: 
                                continue # Guard against empty/corrupted entries
                            
                            # Cache title mappings and document lengths
                            indexed_pages[member.pageid] = member.title
                            doc_lengths[member.pageid] = len(tokens)
                            
                            # Write raw file text to disk so Rocchio can read it later
                            with open(f"{self.data_path}{member.pageid}.txt", "w", encoding="utf-8") as f:
                                f.write(f"TITLE: {member.title}\n{page_data.text}")
                                
                            # Update our frequency tracking dictionary structure
                            for token in tokens:
                                if token not in index: 
                                    index[token] = {}
                                index[token][member.pageid] = index[token].get(member.pageid, 0) + 1
                                
            print(f"Progress checkpoint: Indexed {len(indexed_pages)} total unique articles.")

        # Save to offline storage payload block
        payload = {
            "index": index,
            "doc_lengths": doc_lengths,
            "titles": indexed_pages, # Saves explicit title maps to repair UI rendering
            "num_docs": len(doc_lengths),
            "avg_dl": sum(doc_lengths.values()) / len(doc_lengths) if doc_lengths else 0
        }
        
        with open("index.json", "w") as f:
            json.dump(payload, f)
            
        print(f"\nSuccess! 'data/' directory populated and 'index.json' saved with {len(doc_lengths)} rich entries.")

## Execution
Run the cell below to initiate the crawling and indexing process.

In [16]:
pipeline = WikipediaPipeline()
pipeline.run()

Executing recursive breadth-first traversal across categories...
Progress checkpoint: Indexed 20 total unique articles.
Progress checkpoint: Indexed 20 total unique articles.
Progress checkpoint: Indexed 30 total unique articles.
Progress checkpoint: Indexed 110 total unique articles.
Progress checkpoint: Indexed 119 total unique articles.
Progress checkpoint: Indexed 212 total unique articles.
Progress checkpoint: Indexed 264 total unique articles.
Progress checkpoint: Indexed 304 total unique articles.
Progress checkpoint: Indexed 322 total unique articles.
Progress checkpoint: Indexed 562 total unique articles.
Progress checkpoint: Indexed 609 total unique articles.
Progress checkpoint: Indexed 697 total unique articles.
Progress checkpoint: Indexed 829 total unique articles.
Progress checkpoint: Indexed 1034 total unique articles.
Progress checkpoint: Indexed 1056 total unique articles.
Progress checkpoint: Indexed 1061 total unique articles.
Progress checkpoint: Indexed 1074 total

In [17]:
import json
d = json.load(open("index.json"))
print("docs:", d["num_docs"])          # want ~2000
print("has titles:", "titles" in d)     # want True
print("zero-len docs:", sum(1 for v in d["doc_lengths"].values() if v == 0))  # want 0

docs: 2050
has titles: True
zero-len docs: 0
